# Policy identity, synchronization and the rollout bottleneck

Day 25 · Chapter 14 · CPU worked notebook

Read Chapter14 sections14.6–14.8. Predict: after updating to version3, is a version2 response now a version3 response? Can a faster backward pass dominate the total speedup?

Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Make the prediction above before inspecting the preview.

![Saved reference 1: rollout versions and budget](../figures/chapter-14/day-25-01_rollout_versions_and_budget-01.png)

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


In [ ]:
from dongxi_llms.optimization_diagnostics_lab import RolloutIdentity, accept_rollout, pipeline_budget
row = RolloutIdentity("original-addition-01", 2, "token-v1", "verifier-v1", "group01", 8)
assert accept_rollout(row, 2, "token-v1", "verifier-v1")
try:
    accept_rollout(row, 3, "token-v1", "verifier-v1")
except ValueError as error:
    print("Expected freshness failure:", error)
groups = [2, 4, 8, 16]
budgets = [pipeline_budget(group=g) for g in groups]
fig, ax = plt.subplots()
ax.bar([str(g) for g in groups], [r["projected_generation_seconds"] for r in budgets],
       label="Projected generation")
ax.bar([str(g) for g in groups],
       [r["projected_update_seconds"]-r["projected_generation_seconds"] for r in budgets],
       bottom=[r["projected_generation_seconds"] for r in budgets], label="Learning/verify/sync")
ax.set(xlabel="Rollouts per prompt G", ylabel="Projected synchronous update seconds")
ax.legend()
plt.show()

Reading guide: throughput128 tokens/s is an illustrative control, not a Spark measurement. The same6-second learner becomes a smaller fraction as rollout work grows.

Exercise: intentionally allow lag1, reject a tokenizer mismatch, then compare doubled generation rate with halved learner time.

In [ ]:
# Reference solution: intentional lag contract still preserves behavior version2.
assert accept_rollout(row, 3, "token-v1", "verifier-v1", max_lag=1)
try:
    accept_rollout(row, 2, "token-v2", "verifier-v1")
except ValueError as error:
    print("Expected identity failure:", error)
baseline = pipeline_budget()
faster_generation = pipeline_budget(rollout_tps=256.)
faster_learning = pipeline_budget(learner_seconds=3.)
for name, value in (("baseline", baseline), ("double-generation", faster_generation),
                    ("half-learning-time", faster_learning)):
    print(name, value)
print("Passing metadata checks does not verify the sampling engine's actual weights/log probabilities.")

Controlled change: length128→256 and examine cost. Before engine integration, specify IDs, sampling distribution, tokenizer/template, verifier, version lag and EOS equivalence checks. Evidence boundary: the budget is projected and the version checker is structural, not a production vLLM validation.